# ADA2 LLM runner (conditions A0 to A4)

Runs the five conditions, saves every input and output, and logs model, settings and cost. Use **pilot** mode for rehearsals; **formal** mode is refused until `config.json` is frozen.

**Setup (once):** `pip install -r requirements.txt`, then create a `.env` file in this folder with `OPENAI_API_KEY=...` (see `.env.example`), and set `"model"` in `config.json`.

Nothing here claims what the model will say. Pilot runs are rehearsals and must not be counted in the study.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "runner"))
import importlib, llm_runner as L
importlib.reload(L)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
cfg = L.load_config(ROOT / "config.json")
print("model:", cfg["model"], "| effort:", cfg["reasoning_effort"], "| prompt version:", cfg["prompt_version"], "| frozen:", cfg["frozen"])

model: REPLACE_WITH_MODEL_ID | effort: medium | prompt version: v0.1_DRAFT | frozen: False


## 1. Rebuild the structure evidence (A3) from the saved AlphaFold results

In [2]:
import subprocess
print(subprocess.run([sys.executable, str(ROOT / "runner" / "build_structure_evidence.py")], capture_output=True, text=True, cwd=ROOT).stdout)
display(pd.read_csv(ROOT / "evidence" / "structure_evidence_records_DRAFT.csv")[["id", "species", "type", "value_units", "verification_status"]])

wrote 8 records; 8735 characters



,id,species,type,value_units,verification_status
0,STRUCT-01,human,prediction compared with an experimental structure,RMSD 0.43-0.46 A; Zn-ligand max distance 2.16-2.29 A,DRAFT - not yet checked by a second person
1,STRUCT-02,human and pig,prediction,"ipTM, pTM (0-1); pLDDT (0-100); PAE (A)",DRAFT - not yet checked by a second person
2,STRUCT-03,pig vs human crystal,prediction compared with an experimental structure,RMSD (A); Jaccard (0-1),DRAFT - not yet checked by a second person
3,STRUCT-04,human and pig,calculation (control),RMSD (A),DRAFT - not yet checked by a second person
4,STRUCT-05,human and pig,calculation,RMSD (A),DRAFT - not yet checked by a second person
5,STRUCT-06,human vs pig,calculation,RMSD (A); per-residue distance (A),DRAFT - not yet checked by a second person
6,STRUCT-07,human and pig,prediction,pLDDT (0-100); PAE (A),DRAFT - not yet checked by a second person
7,STRUCT-08,human vs pig,prediction,pLDDT; distance (A),DRAFT - not yet checked by a second person


## 2. Check the inputs (no API call)
Context and system prompt must be identical in every condition, and each evidence package must appear unchanged inside A4. Placeholders show which packages are still missing.

In [3]:
for name, ok, note in L.consistency_checks(cfg):
    print("PASS" if ok else "FAIL", name, f"[{note}]")
rows = []
for c in cfg["conditions"]:
    p = L.build_prompt(cfg, c)
    rows.append(dict(condition=c, packages=cfg["conditions"][c], chars=p["input_chars"], approx_tokens=p["input_chars"] // 4,
                     ids_available=len(p["available_ids"]), placeholders=p["placeholders"], hash=p["input_sha256"][:12]))
pd.DataFrame(rows)

PASS Identical shared context in all conditions [1 distinct context hash(es)]
PASS Identical system prompt in all conditions [1 distinct hash(es)]
PASS sequence package appears unchanged in A4 [verbatim substring check]
PASS surface package appears unchanged in A4 [verbatim substring check]
PASS structure package appears unchanged in A4 [verbatim substring check]
PASS A0 input size (~1751 tokens by chars/4) [limit 100000]
PASS A1 input size (~1812 tokens by chars/4) [limit 100000]
PASS A2 input size (~1813 tokens by chars/4) [limit 100000]
PASS A3 input size (~3910 tokens by chars/4) [limit 100000]
PASS A4 input size (~4059 tokens by chars/4) [limit 100000]


,condition,packages,chars,approx_tokens,ids_available,placeholders,hash
0,A0,[],7007,1751,6,[],3eb642de54ce
1,A1,[sequence],7251,1812,8,[sequence],e42e03b5e7bf
2,A2,[surface],7253,1813,8,[surface],599290da0271
3,A3,[structure],15641,3910,14,[],b3114c509ee0
4,A4,"[sequence, surface, structure]",16239,4059,18,"[sequence, surface]",6b63cf327da3


## 3. Read the exact text one condition will send

In [4]:
CONDITION = "A3"
p = L.build_prompt(cfg, CONDITION)
print(p["user"][:3500], "\n...\n[total", p["input_chars"], "characters]")

Below is the shared background information, followed by any computational evidence available for this task.

=== BACKGROUND INFORMATION ===
Question
Which differences between human and pig ADA2 could explain the reported difference in their activity on free adenosine, and what other functions might be preserved in pig ADA2? We want hypotheses that lead to experiments that distinguish between explanations.

Plain-language definitions
- Protein: a chain of amino acids that folds into a 3D shape. A residue is one amino acid at a position in the chain. A substitution replaces one residue with another.
- Orthologs: corresponding proteins in different species. Human and pig ADA2 are orthologs. Positions must be aligned before they are compared, because insertions and deletions shift the numbering.
- Enzyme and substrate: an enzyme speeds up a chemical reaction; the substrate is the molecule it acts on. The active site is where the chemistry occurs.
- Dimer: two copies of the protein assemble

## 5. Pilot run with the real model (A0 and A3, one repeat each)

Results go to `runs_pilot/`. This is a rehearsal to check length, formatting, logging and cost. Do not count it in the study.
To try another effort level for the pilot: set `cfg["reasoning_effort"] = "high"` and re-run.

In [ ]:
RUN_PILOT = False        # set True to spend API credits
if RUN_PILOT:
    rows = L.run_batch(cfg, ["A0", "A3"], repeats=1, mode="pilot")
    display(pd.DataFrame(rows)[["run_id", "condition", "model_returned", "status", "input_tokens", "output_tokens", "reasoning_tokens", "cost_usd", "unknown_ids"]])

## 6. Read the pilot hypotheses

In [ ]:
for r in L.load_runs("pilot"):
    print("=" * 100, "\n", r["meta"]["run_id"], "|", r["meta"]["model_returned"], "|", r["meta"]["status"])
    for i, h in enumerate(r["output"]["hypotheses"], 1):
        print(f"\n  H{i}: {h['title']}  (confidence: {h['confidence']})")
        print("   Explanation:", h["question_and_explanation"])
        print("   Cites:", h["supporting_evidence_ids"], "| conflicts:", h["conflicting_evidence_ids"])
        print("   Test:", h["proposed_test"])
        print("   Would weaken it:", h["falsifying_outcome"])
        print("   Added value from computation:", h["added_value_from_computation"])

## 7. Citation check
Flags IDs the model cited that were never in its input (invented or from the wrong condition).

In [ ]:
rep = pd.DataFrame(L.citation_report("pilot"))
rep if len(rep) else "no pilot outputs yet"

## 8. Formal batch (later)

Prerequisites, enforced by the code: `"frozen": true`, a real model ID, final (non-DRAFT) prompt version, and no PLACEHOLDER evidence packages. Run 15 formal runs (5 conditions x 3 repeats, random order, seed saved). Do not change prompt or settings after the first formal run; if token limits break comparability, change the common limit and repeat the whole batch.

In [ ]:
RUN_FORMAL = False
if RUN_FORMAL:
    rows = L.run_batch(cfg, list(cfg["conditions"]), repeats=cfg["repeats"], mode="formal")
    display(pd.DataFrame(rows)[["run_id", "condition", "status", "n_hypotheses", "cost_usd", "unknown_ids"]])

## 9. Run log and blind review sheet

In [ ]:
log = ROOT / "runs" / "run_log.csv"
display(pd.read_csv(log) if log.exists() else "no formal runs yet")
# After formal runs: blind sheet for the medical team (no condition labels) plus a key to keep separate
# sheet, key = L.review_sheet("formal")